# [Actividad extracurricular 3b] Módulo logging

## Nombre: Ariel Burgos



### 1. ¿Qué es el módulo `logging`?

El módulo `logging` forma parte de la biblioteca estándar de Python y permite registrar los eventos que ocurren mientras se ejecuta un programa. Estos registros son útiles para detectar errores, conocer el estado de una aplicación y guardar información sobre su funcionamiento.

A diferencia de `print()`, el módulo `logging` permite clasificar los mensajes por niveles, agregar fecha y hora, indicar el archivo y la línea donde ocurrió un evento y enviar los registros a diferentes destinos.

Los componentes principales son:

* **Logger:** crea y administra los mensajes.
* **Handler:** establece dónde se muestran o almacenan los mensajes.
* **Formatter:** determina el formato de cada registro.
* **Filter:** permite seleccionar qué registros serán procesados.

Python ofrece manejadores como `StreamHandler`, para mostrar mensajes en consola, y `FileHandler`, para almacenarlos en archivos. También es posible crear manejadores personalizados, como uno que envíe mensajes a Telegram.

### 2. Niveles de logging

Los niveles principales, de menor a mayor gravedad, son:

| Nivel      | Uso                                             |
| ---------- | ----------------------------------------------- |
| `DEBUG`    | Información detallada para revisar el programa. |
| `INFO`     | Eventos normales de ejecución.                  |
| `WARNING`  | Situaciones que podrían causar problemas.       |
| `ERROR`    | Errores que impiden realizar alguna operación.  |
| `CRITICAL` | Errores graves que podrían detener el programa. |

Cuando se configura un nivel, también se registran los niveles superiores. Por ejemplo, al configurar `WARNING`, se procesan los mensajes `WARNING`, `ERROR` y `CRITICAL`.

### 3. Formato de los registros

El formato puede personalizarse mediante atributos de `LogRecord`. Algunos de los más utilizados son:

* `%(asctime)s`: fecha y hora.
* `%(levelname)s`: nivel del mensaje.
* `%(filename)s`: nombre del archivo.
* `%(lineno)d`: número de línea.
* `%(funcName)s`: nombre de la función.
* `%(name)s`: nombre del logger.
* `%(message)s`: contenido del mensaje.

La documentación oficial indica que `%(asctime)s` permite incluir la fecha y la hora, mientras que `datefmt` permite modificar su presentación.

### 4. Logging en Telegram

Telegram proporciona una API HTTP para que los bots puedan enviar mensajes. El método `sendMessage` permite enviar texto utilizando el token del bot y el identificador del chat.

En este ejemplo, Telegram recibirá solamente mensajes desde el nivel `WARNING`, evitando enviar demasiados mensajes durante la ejecución normal.


In [ ]:
import json
import logging
import os
import sys
import urllib.error
import urllib.parse
import urllib.request


# ==========================================================
# FORMATEADOR CON COLORES PARA LA CONSOLA
# ==========================================================

class FormateadorConColor(logging.Formatter):
    """Agrega colores ANSI según el nivel del registro."""

    RESET = "\033[0m"

    COLORES = {
        logging.DEBUG: "\033[36m",      # Cian
        logging.INFO: "\033[32m",       # Verde
        logging.WARNING: "\033[33m",    # Amarillo
        logging.ERROR: "\033[31m",      # Rojo
        logging.CRITICAL: "\033[1;31m"  # Rojo intenso
    }

    def format(self, record):
        mensaje_original = super().format(record)
        color = self.COLORES.get(record.levelno, self.RESET)

        return f"{color}{mensaje_original}{self.RESET}"


# ==========================================================
# HANDLER PERSONALIZADO PARA TELEGRAM
# ==========================================================

class TelegramHandler(logging.Handler):
    """Envía registros de logging a un chat de Telegram."""

    def __init__(self, token, chat_id, timeout=5):
        super().__init__()

        self.token = token
        self.chat_id = chat_id
        self.timeout = timeout

    def emit(self, record):
        try:
            mensaje = self.format(record)

            # Telegram tiene límites para el tamaño de los mensajes.
            mensaje = mensaje[:4000]

            url = (
                f"https://api.telegram.org/"
                f"bot{self.token}/sendMessage"
            )

            datos = urllib.parse.urlencode({
                "chat_id": self.chat_id,
                "text": mensaje
            }).encode("utf-8")

            solicitud = urllib.request.Request(
                url,
                data=datos,
                method="POST"
            )

            with urllib.request.urlopen(
                solicitud,
                timeout=self.timeout
            ) as respuesta:

                resultado = json.loads(
                    respuesta.read().decode("utf-8")
                )

                if not resultado.get("ok"):
                    raise RuntimeError(
                        "Telegram no pudo enviar el mensaje."
                    )

        except (
            urllib.error.URLError,
            urllib.error.HTTPError,
            TimeoutError,
            RuntimeError,
            ValueError
        ):
            # Evita detener el programa si Telegram no responde.
            self.handleError(record)


# ==========================================================
# CONFIGURACIÓN DEL LOGGER
# ==========================================================

def configurar_logger():
    logger = logging.getLogger("Aplicacion")

    # Nivel mínimo aceptado por el logger.
    logger.setLevel(logging.DEBUG)

    # Evita mensajes repetidos si se llama varias veces.
    logger.handlers.clear()

    # Evita que los mensajes lleguen también al logger raíz.
    logger.propagate = False

    formato = (
        "%(asctime)s | "
        "%(levelname)-8s | "
        "%(filename)s:%(lineno)d | "
        "%(funcName)s() | "
        "%(message)s"
    )

    formato_fecha = "%d/%m/%Y %H:%M:%S"

    # ------------------------------------------------------
    # 1. HANDLER PARA LA CONSOLA
    # ------------------------------------------------------

    consola_handler = logging.StreamHandler(sys.stdout)
    consola_handler.setLevel(logging.DEBUG)

    formato_consola = FormateadorConColor(
        formato,
        datefmt=formato_fecha
    )

    consola_handler.setFormatter(formato_consola)
    logger.addHandler(consola_handler)

    # ------------------------------------------------------
    # 2. HANDLER PARA EL ARCHIVO
    # ------------------------------------------------------

    archivo_handler = logging.FileHandler(
        "registro_aplicacion.log",
        mode="a",
        encoding="utf-8"
    )

    archivo_handler.setLevel(logging.DEBUG)

    formato_archivo = logging.Formatter(
        formato,
        datefmt=formato_fecha
    )

    archivo_handler.setFormatter(formato_archivo)
    logger.addHandler(archivo_handler)

    # ------------------------------------------------------
    # 3. HANDLER PARA TELEGRAM
    # ------------------------------------------------------

    token = os.getenv("TELEGRAM_BOT_TOKEN")
    chat_id = os.getenv("TELEGRAM_CHAT_ID")

    if token and chat_id:
        telegram_handler = TelegramHandler(token, chat_id)

        # Solo se envían advertencias y errores a Telegram.
        telegram_handler.setLevel(logging.WARNING)

        formato_telegram = logging.Formatter(
            "🚨 REGISTRO DEL PROGRAMA\n"
            "Fecha: %(asctime)s\n"
            "Nivel: %(levelname)s\n"
            "Archivo: %(filename)s\n"
            "Línea: %(lineno)d\n"
            "Función: %(funcName)s\n"
            "Mensaje: %(message)s",
            datefmt=formato_fecha
        )

        telegram_handler.setFormatter(formato_telegram)
        logger.addHandler(telegram_handler)

    else:
        logger.warning(
            "Telegram no está configurado. "
            "Faltan TELEGRAM_BOT_TOKEN o TELEGRAM_CHAT_ID."
        )

    return logger


# ==========================================================
# EJEMPLO DE FUNCIONAMIENTO
# ==========================================================

logger = configurar_logger()


def dividir(numero1, numero2):
    logger.debug(
        "Se intenta dividir %s entre %s.",
        numero1,
        numero2
    )

    try:
        resultado = numero1 / numero2

        logger.info(
            "La división se realizó correctamente. Resultado: %s",
            resultado
        )

        return resultado

    except ZeroDivisionError:
        logger.exception(
            "No se puede realizar una división para cero."
        )

        return None


def ejecutar_programa():
    logger.info("El programa ha iniciado.")

    logger.debug(
        "Este es un mensaje utilizado para depuración."
    )

    logger.info(
        "La aplicación está funcionando normalmente."
    )

    logger.warning(
        "Este es un mensaje de advertencia."
    )

    dividir(20, 4)
    dividir(10, 0)

    logger.critical(
        "Ejemplo de un mensaje de nivel crítico."
    )

    logger.info("El programa ha finalizado.")


if __name__ == "__main__":
    ejecutar_programa()